In [22]:
import sys
from pathlib import Path


root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from scripts.preprocess import load_data, prepare_features
from scripts.feature_engineering import TARGET

RANDOM_STATE = 42

train, test, sample = load_data()

X, X_test = prepare_features(train, test)
y = train[TARGET].astype(str)

cat_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# LightGBM menggunakan fitur kategorikal secara native.
X[cat_cols] = X[cat_cols].astype("category")

# Simpan daftar kategori agar tipe dan kategorinya konsisten untuk data test.
category_levels = {
    col: X[col].cat.categories
    for col in cat_cols
}

for col in cat_cols:
    X_test[col] = pd.Categorical(
        X_test[col].astype("string"),
        categories=category_levels[col],
    )

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape)
print("Validasi:", X_valid.shape)
print("Test:", X_test.shape)
print("Fitur kategorikal:", cat_cols)

Train: (426772, 15)
Validasi: (106693, 15)
Test: (228629, 15)
Fitur kategorikal: ['district_id', 'vdcmun_id', 'ward_id', 'land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'superstructure']
